# Construction d'un Système RAG (Retrieval-Augmented Generation) de A à Z



## Architecture globale d'un RAG
Un système RAG résout l'un des principaux problèmes des LLM : l'hallucination et le manque de connaissances fraîches ou privées. Il se déroule en deux grandes phases :

1. **Phase d'Ingestion (Hors-ligne) :** 
   `Document PDF/Texte` ➔ `Extraction de texte` ➔ `Découpage (Chunking)` ➔ `Vectorisation (Embeddings)` ➔ `Stockage (Vector DB)`

2. **Phase de Requête (En-ligne) :** 
   `Question utilisateur` ➔ `Vectorisation de la question` ➔ `Recherche de chunks similaires (Retriever)` ➔ `Construction du Prompt (Question + Chunks)` ➔ `Génération de la réponse par le LLM`

## Étape 0 : Installation et Configuration de l'environnement

### 0.1 Explication des paquets à installer
Pour bâtir notre pipeline, nous installons plusieurs bibliothèques complémentaires :
- `langchain` : Le framework orchestrateur pour lier nos composants (invites, modèles, chaînes).
- `langchain-community` : Contient les intégrations tierces gérées par la communauté (comme les chargeurs de documents et certains modèles).
- `langchain-chroma` : L'intégration officielle pour utiliser ChromaDB comme base de données vectorielle.
- `langchain-huggingface` : L'intégration permettant d'exécuter des modèles d'embeddings de Hugging Face localement.
- `langchain-google-genai` : Le package permettant de s'interfacer avec les modèles de fondation Gemini de Google.
- `pypdf` : Bibliothèque python légère indispensable pour lire et décoder les fichiers PDF.
- `sentence-transformers` : Framework de calcul d'embeddings de phrases basé sur PyTorch, requis en arrière-plan par HuggingFace.

In [28]:
# Option -q (quiet) permet de masquer les logs d'installation verbeux
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-google-genai pypdf sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\vargnomad\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 0.2 Configuration sécurisée de l'API Gemini
Nous allons utiliser le service de LLM managé de Google (Gemini). Pour interagir avec lui via des requêtes HTTP sécurisées, LangChain requiert une clé d'API valide stockée dans les variables d'environnement de votre système d'exploitation.

**Fonctions utilisées :**
- `os.environ` : Un dictionnaire Python qui représente les variables d'environnement du système. Définir une clé dedans permet à toutes les bibliothèques d'y accéder de manière transparente.
- `getpass.getpass()` : Permet d'ouvrir un champ de saisie masqué dans le notebook pour que vous saisissiez votre clé sans qu'elle ne s'affiche en clair à l'écran.

In [29]:
import os
from getpass import getpass

# Vérification et configuration de la variable d'environnement
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Entrez votre clé API Google Gemini : ")
print("Clé API configurée avec succès.")

Clé API configurée avec succès.


---

## Étape 1 : Chargement et Préparation des Données


### 1.1 Explication de la classe `TextLoader` et de `PyPDFLoader`
Pour charger du texte brut ou des PDFs dans LangChain, nous utilisons des **Document Loaders**. Ils ont tous pour mission de lire une source externe et de retourner une liste d'objets `Document`.

#### Qu'est-ce qu'un objet `Document` ?
Un `Document` est une structure de données LangChain contenant deux attributs clés :
1. `page_content` (string) : Le texte brut extrait du fichier.
2. `metadata` (dict) : Un dictionnaire contenant des informations contextuelles (le chemin du fichier, le numéro de page, la date de création, etc.).

#### Comparaison des Chargeurs :
- **`TextLoader(file_path, encoding)`** : Charge un fichier texte simple. 
  - *Paramètre `encoding="utf-8"`* : Indispensable pour éviter les erreurs d'encodage sur les caractères accentués en français.
- **`PyPDFLoader(file_path)`** : Charge un fichier PDF.
  - *Fonctionnement interne* : Il lit le binaire du PDF, utilise `pypdf` pour analyser chaque page séparément, et crée un objet `Document` par page physique du PDF. Cela permet de peupler automatiquement la métadonnée `{'page': <index_page>}`.

#### Méthode utilisée :
- `.load()` : Méthode qui déclenche la lecture effective sur le disque et retourne la liste d'objets `Document`.

In [30]:
from langchain_community.document_loaders import TextLoader

# Instanciation du chargeur de documents
loader = TextLoader("intelligence_artificielle.txt", encoding="utf-8")

# Exécution de la lecture du fichier
documents = loader.load()

print(f"Type retourné par le loader : {type(documents)}")
print(f"Nombre d'objets Document chargés : {len(documents)}")
print(f"Type du premier document : {type(documents[0])}")
print(f"Métadonnées du document : {documents[0].metadata}")

Type retourné par le loader : <class 'list'>
Nombre d'objets Document chargés : 1
Type du premier document : <class 'langchain_core.documents.base.Document'>
Métadonnées du document : {'source': 'intelligence_artificielle.txt'}


### 1.3 Explication approfondie du découpage avec `RecursiveCharacterTextSplitter`
Un document entier est souvent trop volumineux pour être injecté directement dans le prompt d'un LLM. De plus, la recherche vectorielle est beaucoup plus performante sur des morceaux de texte courts et sémantiquement homogènes appelés **chunks**.

Pour découper le texte, LangChain propose la classe `RecursiveCharacterTextSplitter`.

#### Comment fonctionne l'algorithme récursif ?
Contrairement à un découpage naïf (qui couperait par exemple tous les 100 caractères, au milieu d'un mot ou d'une phrase), le `RecursiveCharacterTextSplitter` utilise une liste ordonnée de séparateurs : `["\n\n", "\n", " ", ""]`.
1. Il tente d'abord de découper le texte au niveau des doubles retours à la ligne (paragraphes).
2. Si un paragraphe est toujours plus grand que la taille cible (`chunk_size`), il tente de le découper au niveau des retours à la ligne simples (phrases).
3. Si c'est toujours trop grand, il coupe au niveau des espaces (mots), et en dernier recours, caractère par caractère.
Cela garantit que les paragraphes et les phrases restent autant que possible groupés dans le même chunk.

#### Explication des paramètres du constructeur :
- `chunk_size` (int) : La taille maximale cible de chaque segment (mesurée en nombre de caractères ou de jetons selon la configuration).
- `chunk_overlap` (int) : Le nombre de caractères de chevauchement entre deux chunks successifs. 
  - *Pourquoi ?* Si une information cruciale est écrite à cheval sur la fin du Chunk 1 et le début du Chunk 2, le chevauchement permet de dupliquer cette zone tampon afin que le sens global ne soit pas tronqué.
- `length_function` (callable) : La fonction utilisée pour mesurer la taille du texte. Nous passons la fonction native Python `len` qui compte simplement les caractères.
- `is_separator_regex` (bool) : Indique si les séparateurs doivent être traités comme des expressions régulières (Regex). Ici, défini sur `False` pour un traitement textuel brut.

#### Méthode utilisée :
- `.split_documents(documents)` : Prend en entrée une liste de `Document`, applique l'algorithme de découpage sur le `page_content` de chacun, et renvoie une nouvelle liste de `Document` plus petits. Les métadonnées d'origine sont automatiquement copiées dans chaque sous-document.

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialisation du découpeur récursif avec des paramètres contrôlés
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False
)

# Découpage effectif
chunks = text_splitter.split_documents(documents)

print(f"Le document initial a été subdivisé en {len(chunks)} chunks distincts.\n")

# Visualisons la structure de chaque chunk généré
for i, chunk in enumerate(chunks):
    print(f"[CHUNK {i+1}] (Taille: {len(chunk.page_content)} car.)")
    print(f"Contenu : {repr(chunk.page_content)}")
    print(f"Métadonnées associées : {chunk.metadata}")
    print("-" * 50)

Le document initial a été subdivisé en 19 chunks distincts.

[CHUNK 1] (Taille: 455 car.)
Contenu : "Intelligence Artificielle et Machine Learning : Guide Fondamental\n\n## Qu'est-ce que l'Intelligence Artificielle ?\n\nL'intelligence artificielle (IA) est un domaine de l'informatique qui vise à créer des systèmes capables d'effectuer des tâches qui nécessitent normalement l'intelligence humaine. Ces tâches comprennent la reconnaissance vocale, la prise de décision, la traduction linguistique, la perception visuelle et bien d'autres capacités cognitives."
Métadonnées associées : {'source': 'intelligence_artificielle.txt'}
--------------------------------------------------
[CHUNK 2] (Taille: 327 car.)
Contenu : "L'IA est divisée en deux grandes catégories : l'IA étroite (ou faible), qui est conçue pour effectuer une tâche spécifique comme jouer aux échecs ou recommander des films, et l'IA générale (ou forte), qui serait capable d'effectuer n'importe quelle tâche intellectuelle qu'un êtr

---

## Étape 2 : Génération d'embeddings et Base de Données Vectorielle

### 2.1 Explication de la classe `HuggingFaceEmbeddings`
Un ordinateur ne comprend pas directement le sens des phrases. Nous devons traduire chaque chunk de texte en un vecteur de nombres réels (un tableau de nombres flottants de taille fixe). C'est ce qu'on appelle un **embedding** ou plongement vectoriel.

Nous utilisons la classe `HuggingFaceEmbeddings` qui permet de charger n'importe quel modèle de représentation de phrases hébergé sur la plateforme Hugging Face.

#### Explication du modèle choisi :
- Nous utilisons le modèle `sentence-transformers/all-MiniLM-L6-v2`.
- C'est un modèle extrêmement populaire qui transforme n'importe quel texte en un vecteur de **384 dimensions**.
- Il est compact, très rapide sur CPU et optimisé pour capturer la similarité sémantique (deux phrases exprimant la même idée auront des vecteurs très proches dans l'espace multidimensionnel).

#### Explication des paramètres :
- `model_name` (str) : Le nom du modèle à télécharger depuis Hugging Face.
- `model_kwargs` (dict) : Options de configuration passées au modèle sous-jacent. Ici, nous forçons l'exécution sur le processeur (`'device': 'cpu'`) pour garantir que le code s'exécute chez tous les apprenants sans nécessiter de carte graphique (GPU).

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialisation du modèle d'embeddings open-source local
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("✅ Modèle d'embeddings chargé.")

# Testons le calcul d'un embedding sur une phrase simple
test_vector = embeddings.embed_query("Démo de vectorisation")
print(f"Type du vecteur généré : {type(test_vector)}")
print(f"Longueur (dimension) du vecteur : {len(test_vector)}")
print(f"Aperçu des 5 premières valeurs du vecteur : {test_vector[:5]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7806.25it/s]


✅ Modèle d'embeddings chargé.
Type du vecteur généré : <class 'list'>
Longueur (dimension) du vecteur : 384
Aperçu des 5 premières valeurs du vecteur : [-0.05733300372958183, 0.016543595120310783, 0.004061435349285603, -0.08391281217336655, 0.051641304045915604]


### 2.2 Explication de la classe `Chroma`
Une fois nos chunks convertis en vecteurs, nous devons les stocker dans un index spécialisé capable d'effectuer des calculs géométriques (comme la distance cosinus) à grande vitesse. C'est le rôle de la base de données vectorielle.

Nous utilisons **ChromaDB**, un moteur de base de données vectorielle embarqué (qui s'exécute directement dans la mémoire de notre programme Python ou se sauvegarde dans un simple répertoire local, sans nécessiter l'installation d'un serveur tiers).

#### Explication de la méthode statique `.from_documents(...)` :
C'est la méthode de commodité la plus utilisée pour initialiser la base vectorielle. Elle effectue les opérations suivantes :
1. Elle prend la liste de documents (`chunks`).
2. Elle appelle l'objet `embeddings` pour calculer automatiquement le vecteur de chaque chunk.
3. Elle insère à la fois le texte brut, les métadonnées et le vecteur dans l'index de ChromaDB.

#### Explication des paramètres de la méthode :
- `documents` (List[Document]) : Les fragments de textes générés à l'étape précédente.
- `embedding` (Embeddings) : L'instance de notre modèle de vectorisation, indispensable pour que Chroma sache comment transformer de nouvelles questions en vecteurs.
- `persist_directory` (str) : Le chemin vers un dossier sur le disque. Chroma y stockera ses index de manière persistante (dans une base de données SQLite interne). Si vous redémarrez le notebook, vous pourrez recharger cet index sans avoir besoin de recalculer les vecteurs.

In [33]:
from langchain_chroma import Chroma

# Définition du dossier de sauvegarde
persist_directory = "./chroma_db"

# Construction de l'index et stockage persistant
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Index Chroma créé avec succès et persisté dans le dossier : '{persist_directory}'")

Index Chroma créé avec succès et persisté dans le dossier : './chroma_db'


### 2.3 Explication de la recherche de similarité sémantique
Avant d'implémenter le LLM, nous pouvons interroger directement la base de données vectorielle.

#### Méthode utilisée :
- `.similarity_search(query, k)` :
  - `query` (str) : La question textuelle posée par l'utilisateur.
  - `k` (int) : Le nombre de documents les plus pertinents à renvoyer.
  - *Fonctionnement sous le capot* : Chroma convertit la `query` en vecteur à l'aide du modèle d'embedding, calcule la distance mathématique (similarité cosinus) entre ce vecteur de requête et tous les vecteurs stockés dans la base, puis retourne les `k` documents ayant la distance la plus faible (les plus similaires).

In [34]:
query = "Qu'est-ce que le RAG et comment fonctionne-t-il ?"

# Recherche des k=2 documents les plus proches sémantiquement
retrieved_docs = vectordb.similarity_search(query, k=2)

print(f"Nombre de résultats retournés : {len(retrieved_docs)}\n")
for idx, doc in enumerate(retrieved_docs):
    print(f"--- Extrait {idx+1} ---")
    print(f"Contenu du chunk : {doc.page_content}")
    print(f"Métadonnées associées : {doc.metadata}")
    print()

Nombre de résultats retournés : 2

--- Extrait 1 ---
Contenu du chunk : ## Le Retrieval-Augmented Generation (RAG)

Le RAG est une technique qui combine la recherche d'informations (retrieval) avec la génération de texte par un LLM. Au lieu de se fier uniquement aux connaissances encodées dans les paramètres du modèle lors de l'entraînement, le RAG permet au modèle d'accéder à une base de connaissances externe et à jour.
Métadonnées associées : {'source': 'intelligence_artificielle.txt'}

--- Extrait 2 ---
Contenu du chunk : ## Le Retrieval-Augmented Generation (RAG)

Le RAG est une technique qui combine la recherche d'informations (retrieval) avec la génération de texte par un LLM. Au lieu de se fier uniquement aux connaissances encodées dans les paramètres du modèle lors de l'entraînement, le RAG permet au modèle d'accéder à une base de connaissances externe et à jour.
Métadonnées associées : {'source': 'intelligence_artificielle.txt'}



---

## Étape 3 : Construction du Pipeline RAG avec LangChain

### 3.1 Explication de la méthode `.as_retriever(...)`
Dans l'architecture de LangChain, un **Retriever** est une interface logique simplifiée. Contrairement à une base de données vectorielle complète, un `Retriever` ne sait pas stocker de documents : sa seule et unique fonction est de prendre une chaîne de caractères en entrée (requête) et de retourner une liste de documents.

#### Pourquoi faire cette conversion ?
Parce que les composants de haut niveau de LangChain s'attendent à interagir avec cette interface standardisée, ce qui vous permet de remplacer ChromaDB par n'importe quel autre système de recherche (ElasticSearch, recherche par mots-clés BM25, etc.) sans réécrire votre pipeline.

#### Explication des paramètres :
- `search_kwargs={"k": 2}` : Un dictionnaire d'arguments de configuration passé à l'algorithme de recherche. Ici, nous configurons le retriever pour qu'il ne transmette jamais plus de **2 chunks** au LLM afin de ne pas saturer sa fenêtre de contexte.

In [35]:
# Transformation de la VectorStore en interface Retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

print(f"Type généré : {type(retriever)}")

Type généré : <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


### 3.2 Explication de `ChatPromptTemplate` et des variables système
Le prompt est le moule dans lequel nous fusionnons la question de l'utilisateur et le contexte récupéré depuis ChromaDB avant de soumettre le tout au LLM.

#### Comment structurer un Prompt sécurisé contre l'hallucination ?
Nous utilisons la classe `ChatPromptTemplate` de LangChain pour définir un modèle de conversation structuré.
- **Rôle "system" (System Prompt) :** Configure le comportement, les règles et la personnalité du LLM. Nous lui ordonnons explicitement de répondre uniquement à l'aide des informations fournies sous l'étiquette `{context}`.
- **Rôle "human" :** Représente le message de l'utilisateur final contenant sa question via la variable `{input}`.

#### Explication des variables dynamiques :
- `{context}` : Ce jeton (placeholder) sera automatiquement remplacé par les textes des chunks récupérés par le retriever.
- `{input}` : Ce jeton sera automatiquement remplacé par la question saisie par l'utilisateur.

In [36]:
from langchain_core.prompts import ChatPromptTemplate

# Conception du prompt de contrainte système
system_prompt_instruction = (
    "Vous êtes un expert en intelligence artificielle et en machine learning.\n\n"
    "CONSIGNE CRUCIALE : Répondez UNIQUEMENT en vous basant sur les informations "
    "fournies dans le contexte ci-dessous. Si la réponse ne peut pas être trouvée "
    "dans le contexte, répondez exactement : "
    "'Je ne dispose pas d'informations suffisantes dans ma base de connaissances pour répondre à cette question.'\n\n"
    "Contexte d'appui :\n"
    "{context}"
)

# Assemblage du gabarit d'invite conversationnelle
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt_instruction),
    ("human", "{input}")
])
print("Gabarit de prompt formalisé.")

Gabarit de prompt formalisé.


### 3.3 Explication de la classe `ChatGoogleGenerativeAI`
Pour générer la réponse finale, nous faisons appel au modèle de langage Gemini.

#### Explication des paramètres d'initialisation :
- `model="gemini-2.5-flash-preview-09-2025"` : Spécifie le modèle de langage de la famille Gemini que nous allons interroger via l'API de Google.
- `temperature=0.0` : La température contrôle la créativité et le caractère aléatoire des réponses du modèle.
  - *Pourquoi 0.0 ?* Une valeur proche de `1.0` incite le LLM à être inventif et varié. Une valeur de `0.0` rend le modèle purement déterministe. Il sélectionnera toujours les mots les plus probables et factuels, ce qui est impératif pour une application de type recherche documentaire (RAG) où l'on veut éliminer l'imagination du modèle.

In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Instanciation du grand modèle de langage avec une température nulle pour garantir la factualité
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.0
)

print("Modèle LLM Gemini instancié.")

Modèle LLM Gemini instancié.


### 3.4 Explication des fonctions d'assemblage de chaîne
LangChain offre des utilitaires standardisés modernes pour assembler les composants de notre pipeline.

#### 1. `create_stuff_documents_chain(llm, prompt)`
- **Rôle** : Cette fonction crée une sous-chaîne qui prend une liste de documents (nos chunks récupérés) et les injecte (les "stuff", qui signifie "fourrer/remplir") directement dans la variable `{context}` du gabarit de prompt, puis envoie le prompt complet au LLM.

#### 2. `create_retrieval_chain(retriever, combine_docs_chain)`
- **Rôle** : C'est la chaîne maîtresse. Elle coordonne le flux complet :
  1. Elle intercepte la question de l'utilisateur passée à la clé `"input"`.
  2. Elle envoie automatiquement cette question au `retriever` pour obtenir les documents pertinents.
  3. Elle injecte ces documents sous la clé `"context"` et la question sous la clé `"input"` dans la chaîne de traitement des documents (`combine_docs_chain`).
  4. Elle retourne un dictionnaire complet contenant la question d'origine (`input`), les chunks extraits (`context`), et la réponse rédigée par le LLM (`answer`).

In [38]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# A. Création de la chaîne interne de formatage et d'envoi au LLM
document_chain = create_stuff_documents_chain(llm, prompt_template)

# B. Création de la chaîne externe orchestrant le retriever et la génération
rag_chain = create_retrieval_chain(retriever, document_chain)

print("Pipeline RAG entièrement connecté.")

Pipeline RAG entièrement connecté.


---

## Étape 4 : Tests et Validation du Comportement

### 4.1 Test 1 : Scénario Nominal (Information présente dans la base)
Nous allons poser une question dont la réponse exacte est présente dans le fichier texte initial. 

#### Méthode utilisée :
- `.invoke(dict)` : Déclenche l'exécution synchrone du pipeline. Nous lui passons un dictionnaire contenant la question de l'utilisateur à la clé `"input"`.

In [39]:
question_valide = "Quels sont les trois grands types d'apprentissage automatique ?"

# Exécution du pipeline complet
resultat_nominal = rag_chain.invoke({"input": question_valide})

print(f"Question utilisateur : {question_valide}\n")
print(f"Réponse rédigée par le LLM :\n{resultat_nominal['answer']}\n")

print("--- Diagnostic RAG (Sources exploitées par le système) ---")
for i, doc in enumerate(resultat_nominal['context']):
    print(f"Source n°{i+1} : {doc.page_content} (Fichier: {doc.metadata['source']})")

Question utilisateur : Quels sont les trois grands types d'apprentissage automatique ?

Réponse rédigée par le LLM :
Je ne dispose pas d'informations suffisantes dans ma base de connaissances pour répondre à cette question.

--- Diagnostic RAG (Sources exploitées par le système) ---
Source n°1 : La question de la protection de la vie privée est centrale lorsque les systèmes d'IA traitent de grandes quantités de données personnelles. De plus, l'automatisation liée à l'IA peut transformer profondément le marché du travail, nécessitant des politiques d'adaptation et de reconversion professionnelle. (Fichier: intelligence_artificielle.txt)
Source n°2 : La question de la protection de la vie privée est centrale lorsque les systèmes d'IA traitent de grandes quantités de données personnelles. De plus, l'automatisation liée à l'IA peut transformer profondément le marché du travail, nécessitant des politiques d'adaptation et de reconversion professionnelle. (Fichier: intelligence_artificielle.t

### 4.2 Test 2 : Scénario d'Hallucination Bloquée (Information absente)
Nous posons maintenant une question sans aucun rapport avec les documents. 

Puisque la température est à `0.0` et que nos instructions de prompt sont extrêmement strictes, le LLM doit refuser de répondre et renvoyer exactement la phrase de sécurité définie.

In [40]:
question_inconnue = "Quelle est la recette de la tarte aux pommes normande ?"

# Exécution du pipeline
resultat_inconnu = rag_chain.invoke({"input": question_inconnue})

print(f"Question utilisateur : {question_inconnue}\n")
print(f"Réponse rédigée par le LLM :\n{resultat_inconnu['answer']}")

Question utilisateur : Quelle est la recette de la tarte aux pommes normande ?

Réponse rédigée par le LLM :
Je ne dispose pas d'informations suffisantes dans ma base de connaissances pour répondre à cette question.
